In [1]:
import os
import pandas as pd
from neo4j import GraphDatabase
import json

from dotenv import load_dotenv

### 1. 서버와 연결

In [30]:
# Wait 60 seconds before connecting using these details, or login to https://console.neo4j.io to validate the Aura Instance is available
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI") 
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME") 
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD") 
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") 
AURA_INSTANCEID = os.getenv("AURA_INSTANCEID") 
AURA_INSTANCENAME = os.getenv("AURA_INSTANCENAME") 


In [31]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def close_driver():
    driver.close()

print(driver.get_server_info())

In [24]:
def run_cypher(query: str, params: dict | None = None):
    session_args = {"database": NEO4J_DATABASE}

    with driver.session(**session_args) as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

# 그래프에 어떤 라벨이 있는지(노드 종류)
print(run_cypher("CALL db.labels()"))

# 엣지 타입이 뭐가 있는지
print(run_cypher("CALL db.relationshipTypes()"))

# 노드 5개 샘플 보기
print(run_cypher("MATCH (n) RETURN n LIMIT 5"))

[{'label': 'Keyword'}, {'label': 'Paper'}]
[{'relationshipType': 'PREREQ'}, {'relationshipType': 'ABOUT'}, {'relationshipType': 'IN'}, {'relationshipType': 'REF_BY'}]
[{'n': {'name': 'A priori probability', 'link': 'https://en.wikipedia.org/wiki/A_priori_probability', 'alias': ['a priori probability'], 'categories': ['A priori']}}, {'n': {'name': 'AC-3 algorithm', 'link': 'https://en.wikipedia.org/wiki/AC-3_algorithm', 'alias': ['ac-3', 'AC3', 'ac3', 'A C 3', 'ac 3'], 'categories': ['Constraint programming']}}, {'n': {'name': 'Abstract data type', 'link': 'https://en.wikipedia.org/wiki/Abstract_data_type', 'alias': ['abstract data type', 'ADT', 'abstract datatype', 'abstract data types', 'adt'], 'categories': ['Abstract data types', 'Data types', 'Type theory']}}, {'n': {'name': 'Abstract type', 'link': 'https://en.wikipedia.org/wiki/Abstract_type', 'alias': ['abstract type', 'AbstractType', 'abstract-type', 'abstract type definition', 'abstracttype'], 'categories': ['Type theory']}}, 

### 2. Load data

In [ ]:
## 에이전트 팀에서json 구조를 다르게 전달해주심
## 일단 노션에 나와있는 건 paper_summary, paper_concepts
## 우리에게 필요한 건 paper_name, paper_concepts

In [5]:
BERT = {
    "paper_name": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding", 
    "paper_summary": "BERT(Bidirectional Encoder Representations from Transformers)는 트랜스포머 기반의 심층 양방향 언어 표현 모델로, 양방향 컨텍스트를 동시에 고려하는 사전 학습을 통해 다양한 자연어 이해 작업에서 최첨단 성능을 달성한다. 기존 단방향 언어 모델(예: GPT)과 달리, BERT는 마스크된 언어 모델(MLM)과 다음 문장 예측(NSP) 작업을 통해 사전 학습되며, 미세 조정 시 추가 출력층만 필요로 한다. GLUE, SQuAD, SWAG 등 11개 NLP 작업에서 기존 모델 대비 큰 성능 향상을 보였으며, 모델 크기 증가가 소규모 데이터셋에서도 성능 개선으로 이어짐을 입증했다. 또한, 특징 기반 및 미세 조정 접근법 모두에서 효과적임을 확인했다.",
    "paper_concepts": [
        "Bidirectional Language Model",
        "Masked Language Model (MLM)",
        "Next Sentence Prediction (NSP)",
        "Transformer Encoder",
        "Fine-tuning with Minimal Task-Specific Parameters"
        ]
    }


GSPO = {
    "paper_name": "Group Sequence Policy Optimization",
    "paper_summary": "이 논문은 대규모 언어 모델 훈련을 위한 안정적이고 효율적이며 성능이 우수한 강화 학습 알고리즘인 Group Sequence Policy Optimization(GSPO)을 제안한다. 기존 GRPO 알고리즘의 토큰 수준 중요도 비율 적용 문제를 해결하기 위해 시퀀스 수준 중요도 비율과 클리핑을 도입하여 훈련 안정성과 성능을 개선했다. GSPO는 GRPO 대비 우수한 훈련 효율성과 성능을 보이며, 특히 Mixture-of-Experts(MoE) 모델의 RL 훈련 안정성을 근본적으로 해결하여 복잡한 안정화 전략 없이도 훈련이 가능하다. 또한 RL 인프라 설계 단순화 가능성을 보여주며, 최신 Qwen3 모델의 성능 향상에 기여했다.",
    "paper_concepts": [
        "Group Sequence Policy Optimization",
        "Sequence-level Clipping",
        "Mixture-of-Experts RL",
        "Importance Sampling",
        "Reinforcement Learning Infrastructure"
        ]
    }

### 3. 검색 쿼리
- 고려해야 할 것 -> 매칭할 때 syper에서 대소문자 무시하게끔 해줄 수도 있기 때문에 고려해야 함 (현재는 대소문자 처리에 관해서 신경쓰지 않음)
- **참고 사이트**
    - https://neo4j.com/docs/python-manual/current/query-simple/
    - https://neo4j.com/docs/cypher-manual/current/clauses/match/
- **GDB 스키마 간단**
    - Paper 노드: (:Paper {name, paperId, ...})
    - Keyword 노드: (:Keyword {name, alias: [...] ...})

    - 엣지
        - Paper-[:ABOUT]->Keyword
        - Keyword-[:IN]->Paper
        - Keyword-[:PREREQ]->Keyword
        - Paper-[:REF_BY]->Paper

In [6]:
'''
입력: 논문 제목, 키워드 리스트

- 논문 제목으로 match 시키고
- 키워드 리스트를 받아서 KC와 매치 + alias도 한 번씩 돌고
  - 이때 만약 Exact match의 성능이 별로 좋지 않다면 찾고자 하는 키워드의 단어들을 완벽하게 포함하고 있는 노드를 찾기
- 해당 논문 노드와 키워드 노드와 연결된 모든 그래프를 가져와야 함
  - 이때 너무 많이 가져오면 depth를 제한하거나 하는 방식으로 진행
'''

'\n입력: 논문 제목, 키워드 리스트\n\n- 논문 제목으로 match 시키고\n- 키워드 리스트를 받아서 KC와 매치 + alias도 한 번씩 돌고\n  - 이때 만약 Exact match의 성능이 별로 좋지 않다면 찾고자 하는 키워드의 단어들을 완벽하게 포함하고 있는 노드를 찾기\n- 해당 논문 노드와 키워드 노드와 연결된 모든 그래프를 가져와야 함\n  - 이때 너무 많이 가져오면 depth를 제한하거나 하는 방식으로 진행\n'

In [7]:
def run_cypher(query: str, params: dict | None = None, limit: int = 10):
    params = params or {}
    with driver.session() as session:
        result = session.run(query, params)
        rows = [r.data() for r in result]
    return rows[:limit]

#### 3-1. paper 제목 매칭

In [ ]:
# 1) Exact match **
paper_name = BERT["paper_name"]
q = """
    MATCH (p:Paper {name: $paper_name})
    RETURN p
    """
run_cypher(q, {"paper_name": paper_name})

[{'p': {'citationCount': 108712,
   'year': 2019.0,
   'publication': 'North American Chapter of the Association for Computational Linguistics',
   'referenceCount': 63,
   'name': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
   'abstract': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), BERT is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial task-specific architecture modifications. BERT is conceptually simple and empirically powerful. It obtains ne

In [9]:
# 2) Exact가 잘 안 잡히면: 대소문자 무시 + contains + 인용 수 내림차순

q = """
MATCH (p:Paper)
WHERE toLower(p.name) CONTAINS toLower($paper_name)
RETURN p
ORDER BY p.citationCount DESC
LIMIT 20
"""
run_cypher(q, {"paper_name": paper_name})

[{'p': {'citationCount': 108712,
   'year': 2019.0,
   'publication': 'North American Chapter of the Association for Computational Linguistics',
   'referenceCount': 63,
   'name': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
   'abstract': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), BERT is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial task-specific architecture modifications. BERT is conceptually simple and empirically powerful. It obtains ne

#### 3-2. 키워드 리스트로 매칭

In [10]:
# 1) Exact match

concepts = BERT["paper_concepts"]

q = """
MATCH (kc:Keyword)
WHERE kc.name IN $concepts
RETURN kc
"""
run_cypher(q, {"concepts": concepts})

[]

In [ ]:
# 2) Exact match + 대소문자 무시

concepts = BERT["paper_concepts"]

q = """
MATCH (kc:Keyword)
WHERE toLower(kc.name) IN [c IN $concepts | toLower(c)]
RETURN kc
"""
run_cypher(q, {"concepts": concepts})

[{'kc': {'name': 'bidirectional language model',
   'link': nan,
   'alias': ['bidirectional LM',
    'bi-directional language model',
    'bidirectional language modelling',
    'bi-directional LM'],
   'categories': []}}]

In [ ]:
# 3) 키워드 name 매칭 + Alias 매칭

q = """
MATCH (kc:Keyword)
WHERE kc.name IN $concepts
   OR ANY(a IN coalesce(kc.alias, []) WHERE a IN $concepts)
RETURN kc
"""
# paper에서 뽑힌 키워드 중 Keyword의 name이 하나라도 있는지
# Keyword의 alias 속성 내 리스트 안에 어떤 거라도 paper에서 뽑힌 키워드와 매칭되는 게 있는지
run_cypher(q, {"concepts": concepts})

[]

In [ ]:
# 4) 키워드 name 매칭 + Alias 매칭 + 대소문자 무시 **

q = """
MATCH (kc:Keyword)
WHERE toLower(kc.name) IN [c IN $concepts | toLower(c)]
   OR ANY(a IN coalesce(kc.alias, [])
          WHERE toLower(a) IN [c IN $concepts | toLower(c)])
RETURN kc
"""
# paper에서 뽑힌 키워드 중 Keyword의 name이 하나라도 있는지
# Keyword의 alias 속성 내 리스트 안에 어떤 거라도 paper에서 뽑힌 키워드와 매칭되는 게 있는지
run_cypher(q, {"concepts": concepts})

[{'kc': {'name': 'Transformer-based encoder',
   'link': nan,
   'alias': ['transformer-based encoder',
    'transformer encoder',
    'transformer based encoder'],
   'categories': []}},
 {'kc': {'name': 'bidirectional language model',
   'link': nan,
   'alias': ['bidirectional LM',
    'bi-directional language model',
    'bidirectional language modelling',
    'bi-directional LM'],
   'categories': []}}]

In [ ]:
# 5) 논문에서 뽑힌 키워드들을 띄어쓰기 단위로 쪼개서 Keyword나 alias가 이를 모두 포함한다면 매칭
# GPT에게 도움을 좀 받은거라 이해하고 원하는 대로 되었는지 확인해야 함

q = """
MATCH (kc:Keyword)
WITH kc, $concepts AS concepts
WITH kc,
     [c IN concepts |
        [t IN split(toLower(c), ' ')
         WHERE size(t) > 0]
     ] AS conceptTokens
WITH kc, conceptTokens,
     (coalesce(kc.alias, []) + [kc.name]) AS candidates
WHERE ANY(tokens IN conceptTokens WHERE
          ANY(candidate IN candidates WHERE
              ALL(t IN tokens WHERE toLower(candidate) CONTAINS t)
          )
)
RETURN kc
"""

run_cypher(q, {"concepts": concepts})

# 만약 이게 너무 빡세다면 -> 띄어쓰기를 기준으로 나뉘어졌으 ㄹ때 전체 포함 대신 4개 정도 포함으로 가거나..
# 만약 짧은 단어는 그럼 매칭되는 경우가 너무 많다는 게 걱정이 된다면 -> 짧은 단어는 일단 Exact match로 진행하거나... (즉, 단어 길이 별로 다르게 진행)

[{'kc': {'name': 'BERT (language model)',
   'link': 'https://en.wikipedia.org/wiki/BERT_%28language_model%29',
   'alias': ['bert',
    'BERT model',
    'bert language model',
    'Bidirectional Encoder Representations from Transformers',
    'Bidirectional ERT',
    'bert transformer'],
   'categories': ['2018 in artificial intelligence',
    '2018 software',
    'Google software',
    'Large language models']}},
 {'kc': {'name': 'BERT',
   'link': 'https://en.wikipedia.org/wiki/Bert',
   'alias': ['bert',
    'Bert',
    'Bidirectional Encoder Representations from Transformers',
    'bidirectional encoder representations from transformers',
    'BERT model',
    'bert model'],
   'categories': []}},
 {'kc': {'name': 'BERT model',
   'link': nan,
   'alias': ['bert',
    'bert model',
    'Bidirectional Encoder Representations from Transformers',
    'Bidirectional ERT',
    'BERT'],
   'categories': []}},
 {'kc': {'name': 'Transformer-based encoder',
   'link': nan,
   'alias': ['t

### 3-3. 해당 paper의 ref paper 가져오기

In [21]:
q = """
MATCH (p:Paper {name: $paper_name})
MATCH (ref:Paper)-[r:REF_BY]->(p)
RETURN ref, r
LIMIT 50
"""
run_cypher(q, {"paper_name": paper_name})

[{'ref': {'citationCount': 11976,
   'year': 2018.0,
   'publication': 'North American Chapter of the Association for Computational Linguistics',
   'referenceCount': 64,
   'name': 'Deep Contextualized Word Representations',
   'abstract': 'We introduce a new type of deep contextualized word representation that models both (1) complex characteristics of word use (e.g., syntax and semantics), and (2) how these uses vary across linguistic contexts (i.e., to model polysemy). Our word vectors are learned functions of the internal states of a deep bidirectional language model (biLM), which is pre-trained on a large text corpus. We show that these representations can be easily added to existing models and significantly improve the state of the art across six challenging NLP problems, including question answering, textual entailment and sentiment analysis. We also present an analysis showing that exposing the deep internals of the pre-trained network is crucial, allowing downstream models to

### 3-4. 논문 노드 + 해당 논문의 선행 노드들 + 해당 논문의 fef 논문들 + 매칭된 Keyword 노드 주변 그래프 가져오기

In [22]:
# 1) paper와 직접 연결된 Keyword들(ABOUT/IN) 먼저 확인

q = """
MATCH (p:Paper {name:$paper_name})-[r:ABOUT|IN]-(k:Keyword)
RETURN p, r, k
LIMIT 100
"""
run_cypher(q, {"paper_name": paper_name})

[{'p': {'citationCount': 108712,
   'year': 2019.0,
   'publication': 'North American Chapter of the Association for Computational Linguistics',
   'referenceCount': 63,
   'name': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
   'abstract': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), BERT is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial task-specific architecture modifications. BERT is conceptually simple and empirically powerful. It obtains ne

In [28]:
# 2) 에이전트가 넘겨준 키워드의 선행 개념이 되는 키워드와 그에 관련된 논문 가져오기
#    - 선행 개념이 되는 키워드 -[PREREQ]-> 에이전트가 넘겨준 키워드
#    - 키워드를 제안한 논문 -[ABOUT]-> 에이전트가 넘겨준 키워드

q = """
WITH [c IN $concepts | toLower(c)] AS cs
MATCH (k:Keyword)
WHERE toLower(k.name) IN cs
   OR ANY(a IN coalesce(k.alias, []) WHERE toLower(a) IN cs)

OPTIONAL MATCH (pre:Keyword)-[pr:PREREQ]->(k)
OPTIONAL MATCH (p:Paper)-[ab:ABOUT]->(k)

RETURN k, pre, pr, p, ab
LIMIT 100
"""
run_cypher(q, {"concepts": concepts})

[{'k': {'name': 'Transformer-based encoder',
   'link': nan,
   'alias': ['transformer-based encoder',
    'transformer encoder',
    'transformer based encoder'],
   'categories': []},
  'pre': None,
  'pr': None,
  'p': {'citationCount': 4880,
   'year': 2021.0,
   'publication': 'arXiv.org',
   'referenceCount': 21,
   'name': 'TransUNet: Transformers Make Strong Encoders for Medical Image Segmentation',
   'abstract': 'Medical image segmentation is an essential prerequisite for developing healthcare systems, especially for disease diagnosis and treatment planning. On various medical image segmentation tasks, the u-shaped architecture, also known as U-Net, has become the de-facto standard and achieved tremendous success. However, due to the intrinsic locality of convolution operations, U-Net generally demonstrates limitations in explicitly modeling long-range dependency. Transformers, designed for sequence-to-sequence prediction, have emerged as alternative architectures with innate

In [29]:
# 3) 타겟 논문이 참고한 ref 논문

q = """
MATCH (p:Paper {name: $paper_name})
MATCH (ref:Paper)-[r:REF_BY]->(p)
RETURN ref, r
LIMIT 50
"""
run_cypher(q, {"paper_name": paper_name})

[{'ref': {'citationCount': 11976,
   'year': 2018.0,
   'publication': 'North American Chapter of the Association for Computational Linguistics',
   'referenceCount': 64,
   'name': 'Deep Contextualized Word Representations',
   'abstract': 'We introduce a new type of deep contextualized word representation that models both (1) complex characteristics of word use (e.g., syntax and semantics), and (2) how these uses vary across linguistic contexts (i.e., to model polysemy). Our word vectors are learned functions of the internal states of a deep bidirectional language model (biLM), which is pre-trained on a large text corpus. We show that these representations can be easily added to existing models and significantly improve the state of the art across six challenging NLP problems, including question answering, textual entailment and sentiment analysis. We also present an analysis showing that exposing the deep internals of the pre-trained network is crucial, allowing downstream models to

In [32]:
# 최종

paper_name = BERT["paper_name"]
concepts = BERT["paper_concepts"]

prereq_depth = 2
ref_limit = 30
kw_paper_limit = 50

# strength thresholds (없애고 싶으면 0.0으로)
min_prereq_strength = 0.3
min_kw_strength = 0.3   # ABOUT/IN strength용
min_ref_strength = 0.0  # REF_BY strength용

q = f"""
// 0) 타겟 Paper
MATCH (p:Paper {{name:$paper_name}})

// 1) paper와 직접 연결된 Keyword (ABOUT/IN)
OPTIONAL MATCH (p)-[pk:ABOUT|IN]-(k_paper:Keyword)
WHERE pk.strength IS NULL OR pk.strength >= $min_kw_strength
WITH p,
     collect(DISTINCT {{k:k_paper, r:pk}}) AS paper_kw_links

// 2) agent concepts로 매칭된 seed Keyword (name/alias, case-insensitive)
WITH p, paper_kw_links, [c IN $concepts | toLower(c)] AS cs
MATCH (k_seed:Keyword)
WHERE toLower(k_seed.name) IN cs
   OR ANY(a IN coalesce(k_seed.alias, []) WHERE toLower(a) IN cs)
WITH p, paper_kw_links, collect(DISTINCT k_seed) AS seeds

// 3) seed 주변: prereq (depth 제한) - path로 수집
UNWIND seeds AS s
OPTIONAL MATCH prereqPath = (kpre:Keyword)-[pr:PREREQ*1..{prereq_depth}]->(s)
WHERE ALL(x IN relationships(prereqPath) WHERE x.strength IS NULL OR x.strength >= $min_prereq_strength)
WITH p, paper_kw_links, seeds,
     collect(DISTINCT prereqPath) AS prereq_paths

// 4) seed를 ABOUT한 논문들 (limit)
UNWIND seeds AS s2
OPTIONAL MATCH (p_about:Paper)-[ab:ABOUT]->(s2)
WHERE ab.strength IS NULL OR ab.strength >= $min_kw_strength
WITH p, paper_kw_links, seeds, prereq_paths,
     collect(DISTINCT {{p:p_about, r:ab, k:s2}})[0..$kw_paper_limit] AS seed_about_links

// 5) 타겟 논문이 참고한 ref 논문들 (limit)
OPTIONAL MATCH (ref:Paper)-[rr:REF_BY]->(p)
WHERE rr.strength IS NULL OR rr.strength >= $min_ref_strength
WITH p, paper_kw_links, seeds, prereq_paths, seed_about_links,
     collect(DISTINCT {{p:ref, r:rr}})[0..$ref_limit] AS ref_links

RETURN
  p AS target_paper,
  paper_kw_links,
  seeds AS matched_keywords,
  prereq_paths,
  seed_about_links,
  ref_links
"""

run_cypher(q, {
    "paper_name": paper_name,
    "concepts": concepts,
    "min_prereq_strength": min_prereq_strength,
    "min_kw_strength": min_kw_strength,
    "min_ref_strength": min_ref_strength,
    "ref_limit": ref_limit,
    "kw_paper_limit": kw_paper_limit,
})


# target_paper : Paper 노드 1개
# paper_kw_links : [{k: Keyword노드, r: ABOUT/IN관계}, ...]
# matched_keywords : [Keyword노드, Keyword노드, ...] (seed 목록)
# prereq_paths : [Path, Path, ...] (PREREQ 경로 리스트)
# seed_about_links : [{p: Paper노드, r: ABOUT관계, k: seedKeyword노드}, ...]
# ref_links : [{p: refPaper노드, r: REF_BY관계}, ...]

[{'target_paper': {'citationCount': 108712,
   'year': 2019.0,
   'publication': 'North American Chapter of the Association for Computational Linguistics',
   'referenceCount': 63,
   'name': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
   'abstract': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), BERT is designed to pre-train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a result, the pre-trained BERT model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial task-specific architecture modifications. BERT is conceptually simple and empirically powerful. It